# Padronizacao dos graficos - Counting Sort

Este notebook usa os CSVs gerados pelos benchmarks em Python e Rust para atender ao item 2.3 da especificacao do projeto:

- eixo X: tamanho da entrada (`n`);
- eixo Y: tempo de execucao em segundos;
- sobreposicao da curva teorica esperada;
- uso de escala logaritmica quando ajuda a visualizar diferencas grandes entre linguagens e tamanhos.

Para Counting Sort, a complexidade teorica usada e `T(n, k) = n + k`. Nos experimentos atuais, `k = n`, entao a curva teorica cresce linearmente em relacao a `n`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# O notebook fica em analise/, mas este bloco tambem funciona se for executado da raiz do projeto.
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name.lower() == "analise" else current_dir
analysis_dir = project_root / "analise"
plots_dir = analysis_dir / "graficos"
plots_dir.mkdir(exist_ok=True)

python_csv = analysis_dir / "resultados_python.csv"
rust_csv = analysis_dir / "resultados_rust.csv"

print(f"Lendo: {python_csv}")
print(f"Lendo: {rust_csv}")

In [ ]:
df_python = pd.read_csv(python_csv)
df_rust = pd.read_csv(rust_csv)

df = pd.concat([df_python, df_rust], ignore_index=True)
df["ordenado"] = df["ordenado"].astype(str).str.lower().map({"true": True, "false": False})
df["complexidade_teorica"] = df["n"] + df["k"]

ordem_casos = ["melhor", "medio", "pior"]
ordem_tamanhos = ["pequena", "media", "grande"]
df["caso"] = pd.Categorical(df["caso"], categories=ordem_casos, ordered=True)
df["tamanho_label"] = pd.Categorical(df["tamanho_label"], categories=ordem_tamanhos, ordered=True)

print(f"Total de medicoes: {len(df)}")
print(f"Todas as execucoes ordenaram corretamente? {df['ordenado'].all()}")
df.head()

In [ ]:
resumo = (
    df.groupby(["linguagem", "caso", "tamanho_label", "n", "k"], observed=True)
    .agg(
        media_segundos=("tempo_segundos", "mean"),
        desvio_padrao=("tempo_segundos", "std"),
        minimo=("tempo_segundos", "min"),
        maximo=("tempo_segundos", "max"),
        execucoes=("tempo_segundos", "count"),
        complexidade_teorica=("complexidade_teorica", "first"),
    )
    .reset_index()
)

resumo.to_csv(analysis_dir / "resumo_tempos.csv", index=False)
resumo

## Ajuste da curva teorica

A curva teorica do Counting Sort e proporcional a `n + k`. Como a curva teorica nao esta em segundos por si so, ajustamos uma constante `c` para cada combinacao de linguagem e caso:

`tempo_teorico_ajustado = c * (n + k)`

Esse ajuste permite sobrepor a forma teorica aos tempos reais e verificar se o crescimento observado acompanha a tendencia linear esperada.

In [ ]:
resumo_ajustado = resumo.copy()
resumo_ajustado["constante_c"] = np.nan
resumo_ajustado["tempo_teorico_ajustado"] = np.nan

for (linguagem, caso), indices in resumo_ajustado.groupby(["linguagem", "caso"], observed=True).groups.items():
    grupo = resumo_ajustado.loc[indices]
    x = grupo["complexidade_teorica"].to_numpy(dtype=float)
    y = grupo["media_segundos"].to_numpy(dtype=float)
    c = (x @ y) / (x @ x)
    resumo_ajustado.loc[indices, "constante_c"] = c
    resumo_ajustado.loc[indices, "tempo_teorico_ajustado"] = c * grupo["complexidade_teorica"]

resumo_ajustado.to_csv(analysis_dir / "resumo_tempos_com_teoria.csv", index=False)
resumo_ajustado.head(9)

## Grafico 1 - Tempos reais por linguagem e caso

Este grafico mostra `n` no eixo X e o tempo medio no eixo Y. As barras verticais indicam o desvio-padrao das 30 execucoes. A escala logaritmica ajuda porque Rust e Python ficam em faixas de tempo bem diferentes.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.lineplot(
    data=resumo_ajustado,
    x="n",
    y="media_segundos",
    hue="linguagem",
    style="caso",
    markers=True,
    dashes=False,
    ax=ax,
)

for _, row in resumo_ajustado.iterrows():
    ax.errorbar(
        row["n"],
        row["media_segundos"],
        yerr=row["desvio_padrao"],
        fmt="none",
        color="0.35",
        alpha=0.35,
        capsize=3,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Counting Sort: tempo medio real por tamanho de entrada")
ax.set_xlabel("Tamanho da entrada (n) - escala log")
ax.set_ylabel("Tempo medio de execucao (s) - escala log")
ax.legend(title="Linguagem / caso", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(plots_dir / "01_tempos_reais_log.png", dpi=200, bbox_inches="tight")
plt.show()

## Grafico 2 - Sobreposicao teorica

Cada painel compara os tempos reais com a curva teorica ajustada `c*(n+k)`. Como `k = n` nos testes atuais, a curva esperada e linear em relacao ao tamanho da entrada.

In [ ]:
g = sns.FacetGrid(
    resumo_ajustado,
    row="linguagem",
    col="caso",
    hue="linguagem",
    sharey=False,
    height=3.3,
    aspect=1.15,
)

def plot_real_vs_teorico(data, color, **kwargs):
    data = data.sort_values("n")
    plt.plot(data["n"], data["media_segundos"], marker="o", color=color, label="Real")
    plt.plot(
        data["n"],
        data["tempo_teorico_ajustado"],
        marker="s",
        linestyle="--",
        color="black",
        label="Teorico ajustado: c*(n+k)",
    )
    plt.xscale("log")
    plt.yscale("log")

g.map_dataframe(plot_real_vs_teorico)
g.set_axis_labels("Tamanho da entrada (n) - escala log", "Tempo medio (s) - escala log")
g.set_titles(row_template="{row_name}", col_template="Caso {col_name}")

for ax in g.axes.flat:
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:2], labels[:2], loc="best", fontsize=8)

g.fig.suptitle("Counting Sort: tempos reais vs curva teorica O(n + k)", y=1.03)
g.fig.tight_layout()
g.fig.savefig(plots_dir / "02_real_vs_teorico_log.png", dpi=200, bbox_inches="tight")
plt.show()

## Grafico 3 - Comparacao direta entre Python e Rust

Este grafico usa os tempos medios agregados e compara as duas linguagens no mesmo painel para cada caso. A escala logaritmica no eixo Y facilita a leitura quando uma linguagem e muito mais rapida que a outra.

In [ ]:
g = sns.catplot(
    data=resumo_ajustado,
    kind="bar",
    x="tamanho_label",
    y="media_segundos",
    hue="linguagem",
    col="caso",
    order=ordem_tamanhos,
    col_order=ordem_casos,
    height=4,
    aspect=1,
    errorbar=None,
)

for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.set_xlabel("Tamanho da entrada")
    ax.set_ylabel("Tempo medio (s) - escala log")

g.set_titles("Caso {col_name}")
g.fig.suptitle("Counting Sort: comparacao de tempo medio entre linguagens", y=1.05)
g.fig.tight_layout()
g.fig.savefig(plots_dir / "03_comparacao_linguagens_barras_log.png", dpi=200, bbox_inches="tight")
plt.show()

## Grafico 4 - Speedup Rust vs Python

O speedup e calculado como `tempo_python / tempo_rust`. Valores maiores que 1 indicam quantas vezes Rust foi mais rapido que Python em cada combinacao de tamanho e caso.

In [ ]:
pivot = resumo.pivot_table(
    index=["caso", "tamanho_label", "n"],
    columns="linguagem",
    values="media_segundos",
    observed=True,
).reset_index()

pivot["speedup_rust_vs_python"] = pivot["python"] / pivot["rust"]
pivot.to_csv(analysis_dir / "speedup_rust_vs_python.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.lineplot(
    data=pivot,
    x="n",
    y="speedup_rust_vs_python",
    hue="caso",
    marker="o",
    ax=ax,
)
ax.set_xscale("log")
ax.set_title("Speedup: Python / Rust")
ax.set_xlabel("Tamanho da entrada (n) - escala log")
ax.set_ylabel("Quantas vezes Rust foi mais rapido")
fig.tight_layout()
fig.savefig(plots_dir / "04_speedup_rust_vs_python.png", dpi=200, bbox_inches="tight")
plt.show()

pivot

## Experimento complementar - Variacao de k

Esta secao nao substitui os cenarios exigidos de melhor, medio e pior caso. Ela e um experimento complementar para mostrar uma caracteristica especifica do Counting Sort: o custo tambem depende do intervalo de valores `k`.

Aqui mantemos `n = 100.000` e variamos `k` em tres situacoes:

- `k_pequeno`: `k = 100`;
- `k_igual_n`: `k = 100.000`;
- `k_grande`: `k = 1.000.000`.

In [ ]:
python_k_csv = analysis_dir / "resultados_python_variacao_k.csv"
rust_k_csv = analysis_dir / "resultados_rust_variacao_k.csv"

df_k_python = pd.read_csv(python_k_csv)
df_k_rust = pd.read_csv(rust_k_csv)
df_k = pd.concat([df_k_python, df_k_rust], ignore_index=True)
df_k["ordenado"] = df_k["ordenado"].astype(str).str.lower().map({"true": True, "false": False})
df_k["complexidade_teorica"] = df_k["n"] + df_k["k"]

ordem_k = ["k_pequeno", "k_igual_n", "k_grande"]
df_k["cenario_k"] = pd.Categorical(df_k["cenario_k"], categories=ordem_k, ordered=True)

print(f"Total de medicoes na variacao de k: {len(df_k)}")
print(f"Todas as execucoes ordenaram corretamente? {df_k['ordenado'].all()}")
df_k.head()

In [ ]:
resumo_k = (
    df_k.groupby(["linguagem", "cenario_k", "n", "k"], observed=True)
    .agg(
        media_segundos=("tempo_segundos", "mean"),
        desvio_padrao=("tempo_segundos", "std"),
        minimo=("tempo_segundos", "min"),
        maximo=("tempo_segundos", "max"),
        execucoes=("tempo_segundos", "count"),
        complexidade_teorica=("complexidade_teorica", "first"),
    )
    .reset_index()
)

resumo_k_ajustado = resumo_k.copy()
resumo_k_ajustado["constante_c"] = np.nan
resumo_k_ajustado["tempo_teorico_ajustado"] = np.nan

for linguagem, indices in resumo_k_ajustado.groupby("linguagem", observed=True).groups.items():
    grupo = resumo_k_ajustado.loc[indices]
    x = grupo["complexidade_teorica"].to_numpy(dtype=float)
    y = grupo["media_segundos"].to_numpy(dtype=float)
    c = (x @ y) / (x @ x)
    resumo_k_ajustado.loc[indices, "constante_c"] = c
    resumo_k_ajustado.loc[indices, "tempo_teorico_ajustado"] = c * grupo["complexidade_teorica"]

resumo_k_ajustado.to_csv(analysis_dir / "resumo_variacao_k.csv", index=False)
resumo_k_ajustado

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

sns.lineplot(
    data=resumo_k_ajustado.sort_values("k"),
    x="k",
    y="media_segundos",
    hue="linguagem",
    marker="o",
    ax=ax,
)

for _, row in resumo_k_ajustado.iterrows():
    ax.errorbar(
        row["k"],
        row["media_segundos"],
        yerr=row["desvio_padrao"],
        fmt="none",
        color="0.35",
        alpha=0.35,
        capsize=3,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Counting Sort: impacto da variacao de k com n fixo")
ax.set_xlabel("Intervalo de valores (k) - escala log")
ax.set_ylabel("Tempo medio de execucao (s) - escala log")
fig.tight_layout()
fig.savefig(plots_dir / "05_variacao_k_tempos_log.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

for linguagem, grupo in resumo_k_ajustado.groupby("linguagem", observed=True):
    grupo = grupo.sort_values("k")
    ax.plot(grupo["k"], grupo["media_segundos"], marker="o", label=f"{linguagem} real")
    ax.plot(
        grupo["k"],
        grupo["tempo_teorico_ajustado"],
        marker="s",
        linestyle="--",
        label=f"{linguagem} teorico c*(n+k)",
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Variacao de k: dados reais vs curva teorica O(n + k)")
ax.set_xlabel("Intervalo de valores (k) - escala log")
ax.set_ylabel("Tempo medio de execucao (s) - escala log")
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / "06_variacao_k_real_vs_teorico_log.png", dpi=200, bbox_inches="tight")
plt.show()

## Arquivos gerados

O notebook salva automaticamente:

- `resumo_tempos.csv`: media, desvio-padrao, minimo e maximo por linguagem/caso/tamanho;
- `resumo_tempos_com_teoria.csv`: resumo com a curva teorica ajustada;
- `speedup_rust_vs_python.csv`: comparacao de velocidade entre Python e Rust;
- `resumo_variacao_k.csv`: resumo do experimento complementar variando `k`;
- imagens `.png` em `analise/graficos/`, prontas para usar no relatorio ou nos slides.